## Features engineering

Load the cleaned datasets

In [157]:
import pandas as pd
import numpy as np
from pathlib import Path

CLEANED_MATCHES_PATH = Path("../data/cleaned/tennis_matches.xlsx")
CLEANED_PLAYERS_PATH = Path("../data/cleaned/tennis_players.xlsx")

df_raw = pd.read_excel(CLEANED_MATCHES_PATH)
players_df = pd.read_excel(CLEANED_PLAYERS_PATH)

display(df_raw.head())
display(players_df.head())

,Date,Series,Court,Surface,Round,Tournament,Location,Best of,Winner,Loser,...,W3,L3,W4,L4,W5,L5,B365W,B365L,Wsets,Lsets
0,2005-07-04,International,Outdoor,Clay,1st Round,Swedish Open,Bastad,3.0,robredo t.,tabara m.,...,NaN,NaN,NaN,NaN,NaN,NaN,1.10,6.00,2,0
1,2005-07-04,International,Outdoor,Clay,1st Round,Swedish Open,Bastad,3.0,vinciguerra a.,ryderstedt m.,...,NaN,NaN,NaN,NaN,NaN,NaN,2.10,1.66,2,0
2,2005-07-04,International,Outdoor,Clay,1st Round,Allianz Suisse Open,Gstaad,3.0,verdasco f.,pospisil j.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0
3,2005-07-04,International,Outdoor,Grass,1st Round,Hall of Fame Championships,Newport,3.0,ginepri r.,oudsema s.,...,6.0,0.0,NaN,NaN,NaN,NaN,1.12,5.50,2,1
4,2005-07-04,International,Outdoor,Grass,1st Round,Hall of Fame Championships,Newport,3.0,spadea v.,popp a.,...,6.0,2.0,NaN,NaN,NaN,NaN,2.50,1.50,2,1


,player_key,ioc,dob,hand
0,mulloy g.,USA,1913-11-22,R
1,segura p.,ECU,1921-06-20,R
2,sedgman f.,AUS,1927-10-02,R
3,merlo g.,ITA,1927-10-11,R
4,gonzalez r.,USA,1928-05-09,R


**Randomly winner-loser swap**: Avoid the model learning about choosing always the first player as the winner

In [158]:
df_prepared = df_raw.copy()

# Randomly assign the winner to Player 1 or Player 2.
rng = np.random.default_rng(42)
winner_is_player_1 = rng.random(len(df_prepared)) < 0.5

df_prepared["Player_1"] = np.where(
    winner_is_player_1, df_raw["Winner"], df_raw["Loser"]
)
df_prepared["Player_2"] = np.where(
    winner_is_player_1, df_raw["Loser"], df_raw["Winner"]
)

# Target: 1 if Player 1 wins, otherwise 0.
df_prepared["y"] = winner_is_player_1.astype(int)

# Align player statistics with the randomized player positions.
for name, winner_column, loser_column in [
    ("Rank", "WRank", "LRank"),
    ("Pts", "WPts", "LPts"),
    ("Odds", "B365W", "B365L"),
    ("Sets", "Wsets", "Lsets"),
    ("games1", "W1", "L1"),
    ("games2", "W2", "L2"),
    ("games3", "W3", "L3"),
    ("games4", "W4", "L4"),
    ("games5", "W5", "L5"),
]:
    df_prepared[f"{name}_1"] = np.where(
        winner_is_player_1,
        df_raw[winner_column],
        df_raw[loser_column],
    )
    df_prepared[f"{name}_2"] = np.where(
        winner_is_player_1,
        df_raw[loser_column],
        df_raw[winner_column],
    )
    
df_prepared.head()

,Date,Series,Court,Surface,Round,Tournament,Location,Best of,Winner,Loser,...,games1_1,games1_2,games2_1,games2_2,games3_1,games3_2,games4_1,games4_2,games5_1,games5_2
0,2005-07-04,International,Outdoor,Clay,1st Round,Swedish Open,Bastad,3.0,robredo t.,tabara m.,...,5.0,7.0,0.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2005-07-04,International,Outdoor,Clay,1st Round,Swedish Open,Bastad,3.0,vinciguerra a.,ryderstedt m.,...,6.0,3.0,6.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2005-07-04,International,Outdoor,Clay,1st Round,Allianz Suisse Open,Gstaad,3.0,verdasco f.,pospisil j.,...,2.0,6.0,4.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2005-07-04,International,Outdoor,Grass,1st Round,Hall of Fame Championships,Newport,3.0,ginepri r.,oudsema s.,...,2.0,6.0,7.0,6.0,0.0,6.0,NaN,NaN,NaN,NaN
4,2005-07-04,International,Outdoor,Grass,1st Round,Hall of Fame Championships,Newport,3.0,spadea v.,popp a.,...,4.0,6.0,6.0,3.0,6.0,2.0,NaN,NaN,NaN,NaN


#### Numerical features

**Simple features**: Ranking, points and odds differences

In [159]:
# Get rank, points and odds data
p1_rank, p2_rank = df_prepared["Rank_1"], df_prepared["Rank_2"]
p1_pts, p2_pts = df_prepared["Pts_1"], df_prepared["Pts_2"]
p1_odds, p2_odds = df_prepared["Odds_1"], df_prepared["Odds_2"]

# Standard differences
df_prepared["Rank_Diff"] = p1_rank - p2_rank
df_prepared["Points_Diff"] = p1_pts - p2_pts
df_prepared["Odds_Diff"] = p1_odds - p2_odds

# Logarithmic ratios
df_prepared["Rank_Log_Ratio"] = np.log(p2_rank) - np.log(p1_rank)
df_prepared["Points_Log_Ratio"] = np.log1p(p1_pts) - np.log1p(p2_pts)

valid_odds = (p1_odds > 0) & (p2_odds > 0)
valid_odd1, valid_odd2 = p1_odds[valid_odds], p2_odds[valid_odds]
df_prepared["Odds_Log_Ratio"] = np.nan
df_prepared.loc[valid_odds, "Odds_Log_Ratio"] = np.log(valid_odd1) - np.log(valid_odd2)

In [160]:
numerical_features = [
    "Rank_Diff", "Points_Diff", "Odds_Diff",
    "Rank_Log_Ratio", "Points_Log_Ratio", 
    "Odds_Log_Ratio",
]

Convert **Best of** feature into a binary feature **Best of 5**, which is 1 if the match is best of 5 sets and 0 otherwise.

In [161]:
df_prepared["Best_of_5"] = (df_raw["Best of"] == 5).astype(int)
numerical_features += ["Best_of_5"]

#### Player features

**Age difference**


In [162]:
MIN_PLAYER_AGE, MAX_PLAYER_AGE = 16, 45
AGE_AMBIGUITY_TOLERANCE = 3.0

# Return a dataframe with plausible candidates, one row per match and candidate player
def get_plausible_candidates(df, player_col, extra_cols):
    candidates = players_df[["player_key", "dob"] + extra_cols]
    # left join matches - players datasets
    merged = df[["Date", player_col]].reset_index().merge(
        candidates, left_on=player_col, right_on="player_key", how="left"
    )
    merged["age"] = (merged["Date"] - merged["dob"]).dt.days / 365.25
    return merged[merged["age"].between(MIN_PLAYER_AGE, MAX_PLAYER_AGE)]

def resolve_age(group):
    if len(group) == 1 or (group["age"].max() - group["age"].min()) < AGE_AMBIGUITY_TOLERANCE:
        return group["age"].mean()
    return np.nan

# Return the unique value of an attribute if it is unambiguous, otherwise return NaN
# group: dataframe with plausible candidates for a player in a match
# values: vector of values for the attribute to resolve
def resolve_attribute(group, attribute_col):
    if len(group) == 1 or (group["age"].max() - group["age"].min()) < AGE_AMBIGUITY_TOLERANCE:
        values = group[attribute_col].dropna().unique()
        return values[0] if len(values) == 1 else np.nan
    return np.nan

In [163]:
def compute_age(df, player_col):
    plausible = get_plausible_candidates(df, player_col, [])
    return plausible.groupby("index").apply(resolve_age).reindex(df.index)

age_1 = compute_age(df_prepared, "Player_1")
age_2 = compute_age(df_prepared, "Player_2")
df_prepared["Age_Diff"] = age_1 - age_2
numerical_features += ["Age_Diff"]

coverage = df_prepared["Age_Diff"].notna().mean()
print(f"Age-difference matchups: {coverage:.2%}")

Age-difference matchups: 79.89%


**Hand matchup**

In [164]:
def compute_attribute(df, player_col, attribute_col):
    plausible = get_plausible_candidates(df, player_col, [attribute_col])
    return plausible.groupby("index").apply(resolve_attribute, attribute_col=attribute_col).reindex(df.index)

hand_score = {"L": 1, "R": -1}
hand_1 = compute_attribute(df_prepared, "Player_1", "hand").map(hand_score)
hand_2 = compute_attribute(df_prepared, "Player_2", "hand").map(hand_score)
df_prepared["Hand_Advantage"] = (hand_1 - hand_2) / 2
numerical_features += ["Hand_Advantage"]

coverage = df_prepared["Hand_Advantage"].notna().mean()
print(f"Hand advantage matchups: {coverage:.2%}")

Hand advantage matchups: 77.61%


### Home factor

A player performs better when playing in his home country.

In [165]:
# All cities
print(sorted(df_raw["Location"].unique()))

LOCATION_TO_IOC = {
    "'s-Hertogenbosch": "NED", "Amersfoort": "NED", "Rotterdam": "NED",
    "Acapulco": "MEX", "Los Cabos": "MEX",
    "Adelaide": "AUS", "Brisbane": "AUS", "Melbourne": "AUS", "Sydney": "AUS",
    "Almaty": "KAZ", "Nur-Sultan": "KAZ",
    "Antalya": "TUR", "Istanbul": "TUR",
    "Antwerp": "BEL", "Brussels": "BEL",
    "Athens": "GRE",
    "Atlanta": "USA", "Cincinnati": "USA", "Dallas": "USA", "Delray Beach": "USA",
    "Houston": "USA", "Indian Wells": "USA", "Indianapolis": "USA", "Las Vegas": "USA",
    "Los Angeles": "USA", "Memphis": "USA", "Miami": "USA", "New Haven": "USA",
    "New York": "USA", "Newport": "USA", "San Diego": "USA", "San Jose": "USA",
    "Washington": "USA", "Winston-Salem": "USA",
    "Auckland": "NZL",
    "Bangkok": "THA",
    "Banja Luka": "BIH",
    "Barcelona": "ESP", "Gijon": "ESP", "Madrid": "ESP", "Mallorca": "ESP",
    "Marbella": "ESP", "Valencia": "ESP",
    "Basel": "SUI", "Geneva": "SUI", "Gstaad": "SUI",
    "Bastad": "SWE", "Stockholm": "SWE",
    "Beijing": "CHN", "Chengdu": "CHN", "Hangzhou": "CHN", "Shanghai": "CHN",
    "Shenzhen": "CHN", "Zhuhai": "CHN",
    "Belgrade": "SRB",
    "Bogota": "COL",
    "Bucharest": "ROU",
    "Budapest": "HUN",
    "Buenos Aires": "ARG", "Cordoba": "ARG",
    "Cagliari": "ITA", "Florence": "ITA", "Napoli": "ITA", "Palermo": "ITA",
    "Parma": "ITA", "Rome": "ITA", "Sardinia": "ITA", "Turin": "ITA",
    "Casablanca": "MAR", "Marrakech": "MAR",
    "Chennai": "IND", "Mumbai": "IND", "Pune": "IND",
    "Cologne": "GER", "Dusseldorf": "GER", "Halle": "GER", "Hamburg": "GER",
    "Munich": "GER", "Stuttgart": "GER",
    "Costa Do Sauipe": "BRA", "Rio de Janeiro": "BRA", "Sao Paulo": "BRA",
    "Doha": "QAT",
    "Dubai": "UAE",
    "Eastbourne": "GBR", "London": "GBR", "Nottingham": "GBR", "Queens Club": "GBR",
    "Estoril": "POR", "Oeiras": "POR",
    "Ho Chi Min City": "VIE",
    "Hong Kong": "HKG",
    "Johannesburg": "RSA",
    "Kitzbuhel": "AUT", "Portschach": "AUT", "Vienna": "AUT",
    "Kuala Lumpur": "MAS",
    "Lyon": "FRA", "Marseille": "FRA", "Metz": "FRA", "Montpellier": "FRA",
    "Nice": "FRA", "Paris": "FRA",
    "Monte Carlo": "MON",
    "Montreal": "CAN", "Toronto": "CAN",
    "Moscow": "RUS", "St. Petersburg": "RUS",
    "Quito": "ECU",
    "Santiago": "CHI", "Vina del Mar": "CHI",
    "Seoul": "KOR",
    "Singapore": "SIN",
    "Sofia": "BUL",
    "Sopot": "POL", "Warsaw": "POL",
    "Tel Aviv": "ISR",
    "Tokyo": "JPN",
    "Umag": "CRO", "Zagreb": "CRO",
}

# some cities have end spaces (e.g. "Estoril ")
df_prepared["Location"] = df_prepared["Location"].str.strip()

# Map the location to IOC (e.g. "Rome" -> "ITA")
df_prepared["Location_IOC"] = df_prepared["Location"].map(LOCATION_TO_IOC)

# Get player IOC, resolving ambiguous homonyms the same way as age
ioc_1 = compute_attribute(df_prepared, "Player_1", "ioc")
ioc_2 = compute_attribute(df_prepared, "Player_2", "ioc")

home_1_flag = ioc_1 == df_prepared["Location_IOC"]
home_2_flag = ioc_2 == df_prepared["Location_IOC"]
both_known = ioc_1.notna() & ioc_2.notna() & df_prepared["Location_IOC"].notna()

# Compute the difference in home advantage between Player 1 and Player 2.
df_prepared["Home_Advantage"] = np.where(both_known, home_1_flag.astype(int) - home_2_flag.astype(int), np.nan)
numerical_features += ["Home_Advantage"]

coverage = df_prepared["Home_Advantage"].notna().mean()
print(f"Home advantage matchups: {coverage:.2%}")

["'s-Hertogenbosch", 'Acapulco', 'Adelaide', 'Almaty', 'Amersfoort', 'Antalya', 'Antwerp', 'Athens', 'Atlanta', 'Auckland', 'Bangkok', 'Banja Luka', 'Barcelona', 'Basel', 'Bastad', 'Beijing', 'Belgrade', 'Bogota', 'Brisbane', 'Brussels', 'Bucharest', 'Budapest', 'Buenos Aires', 'Cagliari', 'Casablanca', 'Chengdu', 'Chennai', 'Cincinnati', 'Cologne', 'Cordoba', 'Costa Do Sauipe', 'Dallas', 'Delray Beach', 'Doha', 'Dubai', 'Dubai ', 'Dusseldorf', 'Eastbourne', 'Estoril', 'Estoril ', 'Florence', 'Geneva', 'Gijon', 'Gstaad', 'Halle', 'Hamburg', 'Hangzhou', 'Ho Chi Min City', 'Hong Kong', 'Houston', 'Indian Wells', 'Indianapolis', 'Istanbul', 'Johannesburg ', 'Kitzbuhel', 'Kuala Lumpur', 'Las Vegas', 'London', 'Los Angeles', 'Los Cabos', 'Lyon', 'Madrid', 'Mallorca', 'Marbella', 'Marrakech', 'Marseille', 'Melbourne', 'Memphis', 'Metz', 'Miami', 'Monte Carlo', 'Montpellier', 'Montreal', 'Moscow', 'Mumbai', 'Munich', 'Napoli', 'New Haven', 'New York', 'Newport', 'Nice', 'Nottingham', 'Nur-Sul

#### Advanced features

We can improve the model performance by creating *advanced* features about players ELO ratings, fatigue and head to head statistics.

Firstly, we sort the dataset by date to avoid data leakage

In [166]:
# Elo must be calculated in chronological order.
df_prepared["Date"] = pd.to_datetime(
    df_prepared["Date"],
    errors="raise",
)

# sort dataframe by date to avoid data leakage when calculating Elo ratings
df_prepared = (
    df_prepared
    .sort_values("Date", kind="stable")
    .reset_index(drop=True) # reset index after sorting
)

#### Fatigue

It is computed with the following formula:

$$
\text{EWMA}_{\text{today}} = \text{Load}_{\text{today}} \cdot \lambda + (1 - \lambda) \cdot \text{EWMA}_{\text{yesterday}}, \quad \lambda = \frac{2}{N+1}
$$

The recursive formula is expanded as:

$$
\text{EWMA}_{\text{today}} = (1 - \lambda)^{\text{days\_gap}} \cdot \text{EWMA}_{\text{last\_match}}
$$

since on days without matches, $\text{LOAD}_{\text{today}}$ is 0.

In [167]:
from collections import defaultdict, deque

FATIGUE_WINDOW_DAYS = 5
DEFAULT_REST_DAYS = 30
LAMBDA = 2 / (FATIGUE_WINDOW_DAYS + 1)

player_fatigue_ewma = {}
player_last_date = {}
fatigue_diff = []


def player_fatigue(player, match_date):
    if player not in player_last_date:
        return 0.0
    days_gap = (match_date - player_last_date[player]).days
    return player_fatigue_ewma[player] * (1 - LAMBDA) ** days_gap


def player_rest_days(player, match_date):
    if player not in player_last_date:
        return DEFAULT_REST_DAYS
    return (match_date - player_last_date[player]).days


def update_fatigue(player_1, player_2, match_date):
    fatigue_1 = player_fatigue(player_1, match_date)
    fatigue_2 = player_fatigue(player_2, match_date)
    rest_days_1 = player_rest_days(player_1, match_date)
    rest_days_2 = player_rest_days(player_2, match_date)

    fatigue_diff.append(fatigue_1 - fatigue_2)

    # lambda + (1-lambda) * previous_fatigue
    player_fatigue_ewma[player_1] = LAMBDA + (1 - LAMBDA) * fatigue_1
    player_fatigue_ewma[player_2] = LAMBDA + (1 - LAMBDA) * fatigue_2
    player_last_date[player_1] = match_date
    player_last_date[player_2] = match_date

    return rest_days_1, rest_days_2

#### ELO ratings

Players start with an Elo rating of 1500, and the expected win probability for Player 1 is computed as:

$$ E_1 = \frac{1}{1 + 10^{\frac{R_2 - R_1}{400}}} $$

Ratings are updated post-match via $\Delta R = K \cdot (S_1 - E_1)$, where:

- $R_1, R_2$ are the pre-match Elo ratings of Player 1 and Player 2
- $E_1$ is the expected probability that Player 1 wins
- $S_1$ is the actual result: 1 if Player 1 wins, 0 otherwise
- $K$ controls how strongly ratings are updated

The update factor $K$ dynamically decreases with career matches $M_i(t)$ to reflect growing rating stability:

$$ K_i(t) = \frac{250}{(M_i(t) + 5)^{0.4}} $$

Surface Elo is calculated using the same logic as standard Elo but is tracked independently for each surface. To leverage both, I use a "blended" rating, a 50/50 weighted average of the overall Elo and the surface-specific Elo. This approach stabilizes predictions for surfaces where a player has limited match data by anchoring their performance to their overall ability.

In [168]:
BASE_ELO = 1500.0
SURFACE_BLEND_WEIGHT = 0.5

elo_ratings = {}
surface_elo_ratings = {}
matches_played = {}
matches_played_surface = {}

elo_diff = []
surface_elo_diff = []

def dynamic_k(played):
    return 250 / ((played + 5) ** 0.4)

def blended_surface_rating(overall_rating, surface_rating):
    return (1 - SURFACE_BLEND_WEIGHT) * overall_rating + SURFACE_BLEND_WEIGHT * surface_rating

def expected_score(rating_1, rating_2):
    return 1 / (1 + 10 ** ((rating_2 - rating_1) / 400))

def update_elo_ratings(player_1, player_2, surface, y, elo1, elo2, surf_elo1, surf_elo2):
    blended_1 = blended_surface_rating(elo1, surf_elo1)
    blended_2 = blended_surface_rating(elo2, surf_elo2)

    expected_1 = expected_score(elo1, elo2)
    expected_surface_1 = expected_score(blended_1, blended_2)

    elo_diff.append(elo1 - elo2)
    surface_elo_diff.append(blended_1 - blended_2)

    k1 = dynamic_k(matches_played.get(player_1, 0))
    k2 = dynamic_k(matches_played.get(player_2, 0))
    elo_ratings[player_1] = elo1 + k1 * (y - expected_1)
    elo_ratings[player_2] = elo2 - k2 * (y - expected_1)

    surf_matches_1 = matches_played_surface.get((player_1, surface), 0)
    surf_matches_2 = matches_played_surface.get((player_2, surface), 0)

    k_surf_1 = dynamic_k(surf_matches_1)
    k_surf_2 = dynamic_k(surf_matches_2)
    surface_elo_ratings[(player_1, surface)] = surf_elo1 + k_surf_1 * (y - expected_surface_1)
    surface_elo_ratings[(player_2, surface)] = surf_elo2 - k_surf_2 * (y - expected_surface_1)

    matches_played[player_1] = matches_played.get(player_1, 0) + 1
    matches_played[player_2] = matches_played.get(player_2, 0) + 1
    matches_played_surface[(player_1, surface)] = surf_matches_1 + 1
    matches_played_surface[(player_2, surface)] = surf_matches_2 + 1

#### Recent form

We can store the last 5 matches played by each player using a queue and compute the **recent form** as the average of the last 5 matches played, where a win is $1$ and a loss is $-1$.

E.g. 
- $[1, -1, 1, -1, 1] \implies 0.2$

- $[1] \implies 1.0$

- $[-1, -1, 1] \implies -0.33$

We can also try a weighted recent form, assigning a higher weight to matches against stronger opponents:

$$ \texttt{weighted\_recent\_result} = \texttt{result} \cdot \frac{\texttt{opponent\_ELO}}{\texttt{BASE\_ELO}} $$

In [169]:
RECENT_FORM_WINDOW = 5
recent_results = defaultdict(lambda: deque(maxlen=RECENT_FORM_WINDOW))
recent_form_diff = []

recent_surface_results = defaultdict(lambda: deque(maxlen=RECENT_FORM_WINDOW))
recent_surface_form_diff = []

def update_recent_results(player_1, player_2, surface, y, elo1, elo2, surf_elo1, surf_elo2):
    form_1 = np.mean(recent_results[player_1]) if recent_results[player_1] else 0.0
    form_2 = np.mean(recent_results[player_2]) if recent_results[player_2] else 0.0
    form_surf_1 = np.mean(recent_surface_results[(player_1, surface)]) if recent_surface_results[(player_1, surface)] else 0.0
    form_surf_2 = np.mean(recent_surface_results[(player_2, surface)]) if recent_surface_results[(player_2, surface)] else 0.0

    recent_form_diff.append(form_1 - form_2)
    recent_surface_form_diff.append(form_surf_1 - form_surf_2)

    # result \in {1, -1} for Player 1 win/loss
    sign_1 = 1 if y == 1 else -1
    sign_2 = -sign_1

    recent_results[player_1].append(weighted_result(sign_1, elo2))
    recent_results[player_2].append(weighted_result(sign_2, elo1))
    recent_surface_results[(player_1, surface)].append(weighted_result(sign_1, surf_elo2))
    recent_surface_results[(player_2, surface)].append(weighted_result(sign_2, surf_elo1))

We can also compute the **dominance form**, by considering the number of games won by each player in the last matches, instead of just the match result.

In [170]:
"""Compute number of games won by each player """
games_1 = df_prepared[["games1_1", "games2_1", "games3_1", "games4_1", "games5_1"]]
games_2 = df_prepared[["games1_2", "games2_2", "games3_2", "games4_2", "games5_2"]]

df_prepared["Games_P1"] = games_1.sum(axis=1, skipna=True)
df_prepared["Games_P2"] = games_2.sum(axis=1, skipna=True)

print(df_prepared[["Games_P1", "Games_P2", "Wsets", "Lsets", "y"]].head())

   Games_P1  Games_P2 Wsets Lsets  y
0       5.0      13.0     2     0  0
1      12.0       4.0     2     0  1
2       6.0      12.0     2     0  0
3       9.0      18.0     2     1  0
4      16.0      11.0     2     1  1


In [171]:
DOMINANCE_FORM_WINDOW = 5
dominance_results = defaultdict(lambda: deque(maxlen=DOMINANCE_FORM_WINDOW))
dominance_form_diff = []

def weighted_result(result, opponent_elo):
    return result * (opponent_elo / BASE_ELO)

def update_dominance_form(player_1, player_2, games_won_1, games_total_1, elo1, elo2):
    
    # 1. Compute dominance form average
    dom_1 = np.mean(dominance_results[player_1]) if dominance_results[player_1] else 0.5
    dom_2 = np.mean(dominance_results[player_2]) if dominance_results[player_2] else 0.5

    # 2. Update dominance form
    dominance_form_diff.append(dom_1 - dom_2)

    # 3. Compute ratio of games won after updating dominance, avoiding data leakage
    games_ratio_1 = games_won_1 / games_total_1 if games_total_1 > 0 else 0.5
    games_ratio_2 = 1 - games_ratio_1

    # 4. Update dominance results with weighted result
    dominance_results[player_1].append(weighted_result(games_ratio_1, elo1))
    dominance_results[player_2].append(weighted_result(games_ratio_2, elo2))

#### Head to head statistics

Head-to-head features count previous wins between the same two players, globally and on the current surface. Values are stored before updating the current match to avoid data leakage.

In [172]:
from collections import defaultdict

h2h_balance = defaultdict(int)
h2h_surface_balance = defaultdict(int)

h2h_diff = []
h2h_surface_diff = []


def update_h2h(player_1, player_2, surface, y):
    # (P1, P2) is same as (P2, P1). same key
    pair_key = tuple(sorted((player_1, player_2)))
    surface_key = pair_key + (surface,)
    sign = 1 if player_1 == pair_key[0] else -1

    h2h_diff.append(sign * h2h_balance[pair_key]) # defaultdict inits missing keys to 0
    h2h_surface_diff.append(sign * h2h_surface_balance[surface_key])

    h2h_balance[pair_key] += sign if y == 1 else -sign
    h2h_surface_balance[surface_key] += sign if y == 1 else -sign

In [173]:
for row in df_prepared.itertuples(index=False): # iterates over namedtuples of the dataframe rows
    
    # 1. Get player datas (fullname, date, surface, target)
    player_1, player_2 = row.Player_1, row.Player_2
    match_date, surface = row.Date, row.Surface
    y = row.y
    
    # 2. Get ELOs
    elo1 = elo_ratings.get(player_1, BASE_ELO)
    elo2 = elo_ratings.get(player_2, BASE_ELO)
    surf_elo1 = surface_elo_ratings.get((player_1, surface), BASE_ELO)
    surf_elo2 = surface_elo_ratings.get((player_2, surface), BASE_ELO)

    # 3. Update fatigue
    rest_days_1, rest_days_2 = update_fatigue(player_1, player_2, match_date)

    # 4. Update form and head-to-head statistics
    games_won_P1 = row.Games_P1
    games_total = row.Games_P1 + row.Games_P2
    update_dominance_form(player_1, player_2, games_won_P1, games_total, elo1, elo2)
    update_recent_results(player_1, player_2, surface, y, elo1, elo2, surf_elo1, surf_elo2)
    
    update_h2h(player_1, player_2, surface, y)
    
    # 5. Update ELOs at the end, avoiding data leakage
    update_elo_ratings(player_1, player_2, surface, y, elo1, elo2, surf_elo1, surf_elo2)

advanced_features = {
    "Elo_Diff": elo_diff,
    "Surface_Elo_Diff": surface_elo_diff,
    "Fatigue_Diff": fatigue_diff,
    "Recent_Form_Diff": recent_form_diff,
    "Recent_Surface_Form_Diff": recent_surface_form_diff,
    "Dominance_Form_Diff": dominance_form_diff,
    "H2H_Diff": h2h_diff,
    "H2H_Surface_Diff": h2h_surface_diff,
}

for name, values in advanced_features.items():
    df_prepared[name] = values
    numerical_features.append(name)

# feature correlation
np.corrcoef(surface_elo_diff, elo_diff)[0, 1]

np.float64(0.9316074804629365)

In [174]:
"""Print computed ELO ratings"""

# sort elo ratings and print
sorted(elo_ratings.items(), key=lambda x: x[1], reverse=True)

# filter "djokovic n." surface ratings
for surface in ["Hard", "Clay", "Grass"]:
    print(f"Djokovic N. {surface} Elo: {surface_elo_ratings.get(('djokovic n.', surface), BASE_ELO)}")

Djokovic N. Hard Elo: 2169.2607280533534
Djokovic N. Clay Elo: 1982.8285686361335
Djokovic N. Grass Elo: 2058.2950377273605


#### Categorical features

We create an imputer to manage missing values, using a **most frequent** strategy and **One hot encoding** for low-cardinality categorical features to convert them into numerical ones. 

We manage high cardinality categorical features, such as **Tournament**, by keeping only the most frequent values and grouping the others into a single category called **Other**.

In [175]:
categorical_features = ["Court", "Surface", "Round", "Tournament", "Series"]

#### Splitting and save the dataset

Now we can split the dataset into training and test sets

In [176]:
debug_features = [
    "Date", "Player_1", "Player_2",
    "Odds_1", "Odds_2", # for baseline computation
    "Rank_1", "Rank_2"  # testing accuracy for players with best ranking
]
features = debug_features + numerical_features + categorical_features + ["y"]

df_prepared = df_prepared[features].copy()

And save them as new `xlsx` datasets, to be used in the next notebook for model training and evaluation.

In [177]:
from pathlib import Path
from sklearn.model_selection import train_test_split

TRAINING_PATH = Path("../data/training/tennis_training.xlsx")
TESTING_PATH = Path("../data/testing/tennis_testing.xlsx")

train_df, test_df = train_test_split(
    df_prepared,
    train_size=0.7,
    shuffle=False
)

TRAINING_PATH.parent.mkdir(parents=True, exist_ok=True)
train_df.to_excel(TRAINING_PATH, index=False)

TESTING_PATH.parent.mkdir(parents=True, exist_ok=True)
test_df.to_excel(TESTING_PATH, index=False)